# Lumine Mini 2B, LFM3
Short fresh test: 1,000 random targets, 1,000 optimizer steps, eight action tokens, 10% synthetic idle bootstrap. With effective batch 8 this repeats the subset for 8 epochs. Evaluation remains the fixed 256 validation targets. Remove --max-steps and --subset for the full configured epoch. Set --bootstrap-prob 0 to disable idle-start augmentation. Use a unique output directory for each fresh run. HF_TOKEN must be a Colab secret with dataset and bucket access.


In [ ]:
import subprocess, sys
from pathlib import Path
if not Path('/content/Lumine/.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/koboshchan/Lumine.git', '/content/Lumine'], check=True)
subprocess.run(['git', '-C', '/content/Lumine', 'fetch', 'origin', 'main'], check=True)
REPO_COMMIT = subprocess.check_output(['git', '-C', '/content/Lumine', 'rev-parse', 'origin/main'], text=True).strip()
subprocess.run(['git', '-C', '/content/Lumine', 'checkout', '--detach', REPO_COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', '/content/Lumine/training/lfm3/requirements.txt'], check=True)


In [ ]:
import os, json
from google.colab import userdata
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
assert os.environ['HF_TOKEN'], 'Enable HF_TOKEN in Colab Secrets'
from huggingface_hub import snapshot_download
config = json.load(open('/content/Lumine/training/lfm3/config.json'))
snapshot_download(config['dataset'], repo_type='dataset', revision=config['dataset_revision'], local_dir='/content/lumine-data', allow_patterns=['data/*.parquet', 'resize-manifest.json'], max_workers=2)


In [ ]:
command = [sys.executable, '-u', '/content/Lumine/training/lfm3/train.py',
    '--dataset-dir', '/content/lumine-data', '--output', '/content/lumine-lfm3-test-1k',
    '--workers', '8', '--max-steps', '1000', '--subset', '1000', '--subset-seed', '42',
    '--action-special-tokens', '--bootstrap-prob', '0.1']
# Stream output rather than hiding Trainer progress behind subprocess.run.
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, command)
